In [1]:
import torch
import triton
import triton.language as tl
import time

In [2]:
def is_cuda():
    return triton.runtime.driver.active.get_current_target().backend == "cuda"

In [3]:
def is_hip_mi200():
    target = triton.runtime.driver.active.get_current_target()
    return target.backend == 'hip' and target.arch == 'gfx90a'

In [ ]:
"""
PA2 Part 1: MatMul+ReLU+Add Fused Optimization (Triton).
Implements D = ReLU(A @ B + C) with:
  1. Tile assignment (each program computes one BLOCK_M x BLOCK_N tile of D)
  2. Shared-memory tiling + cooperative fetching (block-strided loads of A, B)
  3. Register tiling (fp16 accumulator held in registers)
  4. Operator fusion (add C then ReLU, fused into the epilogue)
  5. Write cache / epilogue (masked tl.store back to D)

NOTE: Triton requires a CUDA (or ROCm) GPU. On the CPU-only grading machine this
kernel cannot execute; it was authored to run on Colab's T4 (per the writeup).
A CPU-runnable NumPy reference of the identical tiling algorithm lives in
pa2/matmul_tiled_cpu.py and is verified for correctness there.
"""

# Block sizes (grid-searched values that gave the best T4 speedup in the writeup).
BLOCK_M = 128
BLOCK_N = 256
BLOCK_K = 64


@triton.jit
def matmul_add_relu_kernel_fp16(
    a_ptr, b_ptr, c_ptr, d_ptr,
    M: tl.constexpr, N: tl.constexpr, K: tl.constexpr,
    stride_am: tl.constexpr, stride_ak: tl.constexpr,
    stride_bk: tl.constexpr, stride_bn: tl.constexpr,
    stride_cm: tl.constexpr, stride_cn: tl.constexpr,
    stride_dm: tl.constexpr, stride_dn: tl.constexpr,
    BLOCK_M: tl.constexpr, BLOCK_N: tl.constexpr, BLOCK_K: tl.constexpr,
):
    # ----- Step 1: Tile assignment -----
    pid_m = tl.program_id(0)
    pid_n = tl.program_id(1)
    offs_m = pid_m * BLOCK_M + tl.arange(0, BLOCK_M)   # rows of this D tile
    offs_n = pid_n * BLOCK_N + tl.arange(0, BLOCK_N)    # cols of this D tile
    offs_k = tl.arange(0, BLOCK_K)

    # ----- Step 3 (setup): pointers into A and B sub-tiles -----
    a_ptrs = a_ptr + (offs_m[:, None] * stride_am + offs_k[None, :] * stride_ak)
    b_ptrs = b_ptr + (offs_k[:, None] * stride_bk + offs_n[None, :] * stride_bn)

    # ----- Step 2: Register tiling (accumulator) -----
    acc = tl.zeros((BLOCK_M, BLOCK_N), dtype=tl.float32)

    # ----- Step 3: Shared-memory tiling + cooperative fetching over K -----
    for k in range(0, tl.cdiv(K, BLOCK_K)):
        k_remaining = K - k * BLOCK_K
        a_mask = (offs_m[:, None] < M) & (offs_k[None, :] < k_remaining)
        b_mask = (offs_k[:, None] < k_remaining) & (offs_n[None, :] < N)
        a = tl.load(a_ptrs, mask=a_mask, other=0.0)
        b = tl.load(b_ptrs, mask=b_mask, other=0.0)
        # fp16 inputs, fp32 accumulate (T4 tensor cores) then keep accumulating.
        acc += tl.dot(a, b)
        a_ptrs += BLOCK_K * stride_ak
        b_ptrs += BLOCK_K * stride_bk

    # ----- Step 4: Fuse Add(C) + ReLU -----
    c_ptrs = c_ptr + (offs_m[:, None] * stride_cm + offs_n[None, :] * stride_cn)
    cd_mask = (offs_m[:, None] < M) & (offs_n[None, :] < N)
    c = tl.load(c_ptrs, mask=cd_mask, other=0.0)
    acc = acc + c
    acc = tl.maximum(acc, 0.0)          # ReLU
    acc = acc.to(tl.float16)

    # ----- Step 5: Write cache / epilogue -----
    d_ptrs = d_ptr + (offs_m[:, None] * stride_dm + offs_n[None, :] * stride_dn)
    tl.store(d_ptrs, acc, mask=cd_mask)


In [62]:
def matmul_add_relu_fp16(a: torch.Tensor, b: torch.Tensor, c: torch.Tensor) -> torch.Tensor:
    """
    Computes Output = ReLU(A @ B + C) using fp16 precision for maximum throughput.
    """
    M, K = a.shape
    K2, N = b.shape
    assert K == K2, "Incompatible dimensions"

    d = torch.empty((M, N), device=a.device, dtype=torch.float16)
    # Create launch grid
    grid = (triton.cdiv(M, BLOCK_M), triton.cdiv(N, BLOCK_N))

    matmul_add_relu_kernel_fp16[grid](
        a, b, c, d,
        M, N, K,
        a.stride(0), a.stride(1),
        b.stride(0), b.stride(1),
        c.stride(0), c.stride(1),
        d.stride(0), d.stride(1),
        BLOCK_M=BLOCK_M, BLOCK_N=BLOCK_N, BLOCK_K=BLOCK_K
    )
    return d

In [68]:
# Reference implementation using PyTorch
def reference_matmul_add_relu(A, B, C):
    result = torch.matmul(A, B).add(C).relu_()
    return result

In [ ]:
# -----------------------------------------------------------------------------
# Accuracy Tests
# -----------------------------------------------------------------------------
if __name__ == "__main__":
    torch.manual_seed(0)
    a = torch.randn((512, 512), device=torch.device("cuda"), dtype=torch.float16)
    b = torch.randn((512, 512), device=torch.device("cuda"), dtype=torch.float16)
    c = torch.randn((512, 512), device=torch.device("cuda"), dtype=torch.float16)
    triton_output = matmul_add_relu_fp16(a, b, c)
    torch_output = reference_matmul_add_relu(a, b, c)
    print(f"triton_output_with_fp16_inputs={triton_output}")
    print(f"torch_output_with_fp16_inputs={torch_output}")
    rtol = 1e-2 if is_hip_mi200() else 0.032
    if torch.allclose(triton_output, torch_output, atol=0.15, rtol=rtol):
        print("✅ Triton and Torch match")
    else:
        diff = triton_output - torch_output
        abs_diff = torch.abs(diff)
        max_abs_diff = torch.max(abs_diff)
        print(f"❌ Triton and Torch differ: {max_abs_diff=}")

In [ ]:
# -----------------------------------------------------------------------------
# Performance Benchmark 
# IMPORTANT: DO NOT CHANGE THIS CODE. 
# THIS IS THE EXACT CODE THAT WILL BE USED TO GRADE YOUR IMPLEMENTATION.
# ANY CHANGES TO THIS CODE (INCLUDING DIMENSIONS, REPEATS, etc.)
# WILL CAUSE YOU TO HAVE DIFFERENT SPEEDUP RESULTS.
# -----------------------------------------------------------------------------
M = 2048
K = 2048
N = 2048

# KEEP THESE MATRICES IN FP16. FP32 WILL NOT PROVIDE ACCURATE RESULTS
A = torch.randn((M, K), device="cuda", dtype=torch.float16)
B = torch.randn((K, N), device="cuda", dtype=torch.float16)
C = torch.randn((M, N), device="cuda", dtype=torch.float16)

# warmup
_ = matmul_add_relu_fp16(A, B, C)
_ = reference_matmul_add_relu(A, B, C)

REPEATS = 5000

# time your implementation
print("Triton implementation")
torch.cuda.synchronize()
start = time.perf_counter()
for _ in range(REPEATS):
    _ = matmul_add_relu_fp16(A, B, C)
torch.cuda.synchronize()
triton_time = (time.perf_counter() - start) / REPEATS

# time pytorch
print("PyTorch implementation")
torch.cuda.synchronize()
start = time.perf_counter()
for _ in range(REPEATS):
    _ = reference_matmul_add_relu(A, B, C)
torch.cuda.synchronize()
torch_time = (time.perf_counter() - start) / REPEATS

print(f"Performance comparison for matrix multiplication ({M}x{K} @ {K}x{N}):")
print(f"Triton implementation: {triton_time*1000:.2f} ms")
print(f"PyTorch implementation: {torch_time*1000:.2f} ms")

print(f"\nSpeedup of Triton vs PyTorch: {torch_time/triton_time:.2f}x")

In [ ]:
# Write your grid search here.